<a href="https://colab.research.google.com/github/flash-berry/MachineLearning-2025/blob/main/HW_4_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Домашнее задание 4**

ДЗ#4

Эмулировать изменение инструкции асессорской разметки.
Взять эмуляцию асессоров из ДЗ#3, разделить их на две группы - контроль и тест. Во второй группе поменять распределение оценок у части асессоров, выбрать статистический тест и проверить, что изменения статзначимы.

Дедлайн: 22 ноября

---



**Выполнил студент:** Пышный Артём



---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kruskal

Перенесём код из ДЗ#3

In [ ]:
def probs_dict(n_positions, n_labels):
    probs_prior = np.repeat(1e-3, n_labels)
    probs_posterior = {pos: probs_prior.copy() for pos in range(1, n_positions + 1)}

    for pos in range(1, n_positions + 1):
        pos_data = df_gt[df_gt['position'] == pos]['label']

        label_counts = np.zeros(n_labels)
        for label in range(n_labels):
            label_counts[label] = (pos_data == label).sum()

        probs_posterior[pos] = probs_prior + label_counts

    for pos in range(1, n_positions + 1):
        probs_posterior[pos] = probs_posterior[pos] / probs_posterior[pos].sum()

    return probs_posterior

def generate_assessor_labels(df_gt, probs, n_assessors, fraud_ratio, bad_ratio):
    assessors = [f"A{i + 1}" for i in range(n_assessors)]
    n_fraud = int(n_assessors * fraud_ratio)
    n_bad = int(n_assessors * bad_ratio)
    n_good = n_assessors - n_fraud - n_bad

    types = (
        ["good"] * n_good +
        ["bad"] * n_bad +
        ["fraud"] * n_fraud
    )
    np.random.shuffle(types)

    data = []

    for (task, true_label, position) in df_gt.values:
        for j, a_type in zip(assessors, types):
            if a_type == "good":
                proba = probs[position]
                label = np.random.choice(classes, p = proba)
            elif a_type == "bad":
                label = np.random.choice(classes)
            else:
                label = np.random.choice(classes, p = proba[::-1])

            data.append([task, j, position, label, a_type])

    df = pd.DataFrame(data, columns=['task', 'worker', 'position', 'label', 'type'])

    return df

In [ ]:
# 5 queries for evaluation
query_1 = "какого цвета глаза у ким кардашьян?"
query_2 = "из чего была сделана куртка буратино"
query_3 = "когда вышел айфон 15"
query_4 = "как заменить шланг на бойлере не сливая воду"
query_5 = "чем полезен зелёный чай с мелиссой"

num_results = 5

alpha = 0.05

# ground truth
gt = [
    ['https://clck.ru/3PkFx7', 2, 1],
    ['https://clck.ru/3PkFy9', 2, 2],
    ['https://clck.ru/3PkFzN', 1, 3],
    ['https://clck.ru/3PkG2u', 0, 4],
    ['https://clck.ru/3PkGL2', 1, 5],

    ['https://clck.ru/3PkGTG', 2, 1],
    ['https://clck.ru/3PkGej', 2, 2],
    ['https://clck.ru/3PkJeU', 2, 3],
    ['https://clck.ru/3PkJgq', 1, 4],
    ['https://clck.ru/3PkJsf', 0, 5],

    ['https://clck.ru/3PkKSs', 2, 1],
    ['https://clck.ru/3PkKHW', 2, 2],
    ['https://clck.ru/3PkKag', 1, 3],
    ['https://clck.ru/3PkKjU', 0, 4],
    ['https://clck.ru/3PkfLu', 0, 5],

    ['https://clck.ru/3PkfRe', 2, 1],
    ['https://clck.ru/3Pkfpr', 2, 2],
    ['https://clck.ru/3PkfsJ', 2, 3],
    ['https://clck.ru/3PkfuH', 0, 4],
    ['https://clck.ru/3PkfyJ', 1, 5],

    ['https://clck.ru/3Pkg5G', 1, 1],
    ['https://clck.ru/3Pkg6m', 2, 2],
    ['https://clck.ru/3Pkg8g', 2, 3],
    ['https://clck.ru/3PkgAx', 2, 4],
    ['https://clck.ru/3PkgCA', 1, 5]
]

columns_gt = ['task', 'label', 'position']

df_gt = pd.DataFrame(gt, columns=columns_gt)

df_gt.head()

,task,label,position
0,https://clck.ru/3PkFx7,2,1
1,https://clck.ru/3PkFy9,2,2
2,https://clck.ru/3PkFzN,1,3
3,https://clck.ru/3PkG2u,0,4
4,https://clck.ru/3PkGL2,1,5


In [ ]:
classes = df_gt["label"].sort_values().unique()
probs = probs_dict(num_results, len(classes))

for position in range(1, num_results + 1):
    proba_pos = probs[position]
    print(f"\n\tПозиция: {position}")
    print(f"Наиболее вероятный label: {proba_pos.argmax()} (p = {proba_pos.max():.4f})")

    for label in range(len(classes)):
        print(f"label {label}: {proba_pos[label]:.4f}")


	Позиция: 1
Наиболее вероятный label: 2 (p = 0.7997)
label 0: 0.0002
label 1: 0.2001
label 2: 0.7997

	Позиция: 2
Наиболее вероятный label: 2 (p = 0.9996)
label 0: 0.0002
label 1: 0.0002
label 2: 0.9996

	Позиция: 3
Наиболее вероятный label: 2 (p = 0.5998)
label 0: 0.0002
label 1: 0.4000
label 2: 0.5998

	Позиция: 4
Наиболее вероятный label: 0 (p = 0.5998)
label 0: 0.5998
label 1: 0.2001
label 2: 0.2001

	Позиция: 5
Наиболее вероятный label: 1 (p = 0.5998)
label 0: 0.4000
label 1: 0.5998
label 2: 0.0002


Генерируем 2 группы для проведения A/B теста:



*   Control - в которой будут асессоры с правильным распределением оценок
*   Test - в которой часть асессоров будет с другим распределением (эмулируем изменение инструкции асессорской разметки)



In [ ]:
control = generate_assessor_labels(df_gt, probs, n_assessors = 1000, fraud_ratio = 0, bad_ratio = 0)
test = generate_assessor_labels(df_gt, probs, n_assessors = 1000, fraud_ratio = 0.1, bad_ratio = 0.1)

Для оценки статзначимости изменений был выбран статистическйи тест Краскела-Уоллиса, который подходит для порядковых шкал.

In [ ]:
stat, p_value = kruskal(control['label'], test['label'])

print(f"stat = {stat}, p_value = {p_value}")
if p_value < alpha:
  print(f"Изменения статзначимы на уровне значимости {alpha}")
else:
  print(f"Изменения не статзначимы на уровне значимости {alpha}")

stat = 165.92095494154643, p_value = 5.756157867317543e-38
Изменения статзначимы на уровне значимости 0.05
